In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

import pandas as pd
import numpy as np
import time

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [12]:
df = pd.read_csv("Data/train.csv")

y = df["label"].values
X = df.drop("label", axis=1).values

X = X / 255.0
X = X.reshape(-1, 1, 28, 28)

# Resize to 32x32 for GoogLeNet compatibility
X_tensor = torch.tensor(X, dtype=torch.float32)
X_tensor = F.interpolate(X_tensor, size=(32, 32))

y_tensor = torch.tensor(y, dtype=torch.long)

dataset = TensorDataset(X_tensor, y_tensor)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [13]:
class InceptionBlock(nn.Module):
    def __init__(self, in_channels, c1, c3_reduce, c3, c5_reduce, c5, pool_proj):
        super().__init__()
        
        # 1x1
        self.branch1 = nn.Conv2d(in_channels, c1, kernel_size=1)
        
        # 1x1 -> 3x3
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, c3_reduce, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(c3_reduce, c3, kernel_size=3, padding=1)
        )
        
        # 1x1 -> 5x5
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, c5_reduce, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(c5_reduce, c5, kernel_size=5, padding=2)
        )
        
        # pool -> 1x1
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_channels, pool_proj, kernel_size=1)
        )
        
    def forward(self, x):
        return torch.cat([
            self.branch1(x),
            self.branch2(x),
            self.branch3(x),
            self.branch4(x)
        ], dim=1)

In [14]:
class InceptionBlock_BN(nn.Module):
    def __init__(self, in_channels, c1, c3_reduce, c3, c5_reduce, c5, pool_proj):
        super().__init__()
        
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, c1, 1),
            nn.BatchNorm2d(c1),
            nn.ReLU()
        )
        
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, c3_reduce, 1),
            nn.BatchNorm2d(c3_reduce),
            nn.ReLU(),
            nn.Conv2d(c3_reduce, c3, 3, padding=1),
            nn.BatchNorm2d(c3),
            nn.ReLU()
        )
        
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, c5_reduce, 1),
            nn.BatchNorm2d(c5_reduce),
            nn.ReLU(),
            nn.Conv2d(c5_reduce, c5, 5, padding=2),
            nn.BatchNorm2d(c5),
            nn.ReLU()
        )
        
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_channels, pool_proj, 1),
            nn.BatchNorm2d(pool_proj),
            nn.ReLU()
        )
        
    def forward(self, x):
        return torch.cat([
            self.branch1(x),
            self.branch2(x),
            self.branch3(x),
            self.branch4(x)
        ], dim=1)

In [15]:
class InceptionBlock_Modified(nn.Module):
    def __init__(self, in_channels, c1, c3_reduce, c3, pool_proj):
        super().__init__()
        
        # 1x1
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, c1, 1),
            nn.BatchNorm2d(c1),
            nn.ReLU()
        )
        
        # 1x1 -> 3x3 -> 3x3 (instead of 5x5)
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, c3_reduce, 1),
            nn.BatchNorm2d(c3_reduce),
            nn.ReLU(),
            nn.Conv2d(c3_reduce, c3, 3, padding=1),
            nn.BatchNorm2d(c3),
            nn.ReLU(),
            nn.Conv2d(c3, c3, 3, padding=1),
            nn.BatchNorm2d(c3),
            nn.ReLU()
        )
        
        # Factorized conv (1x3 + 3x1)
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, c3_reduce, 1),
            nn.BatchNorm2d(c3_reduce),
            nn.ReLU(),
            nn.Conv2d(c3_reduce, c3, (1,3), padding=(0,1)),
            nn.BatchNorm2d(c3),
            nn.ReLU(),
            nn.Conv2d(c3, c3, (3,1), padding=(1,0)),
            nn.BatchNorm2d(c3),
            nn.ReLU()
        )
        
        self.branch4 = nn.Sequential(
            nn.AvgPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_channels, pool_proj, 1),
            nn.BatchNorm2d(pool_proj),
            nn.ReLU()
        )
        
    def forward(self, x):
        return torch.cat([
            self.branch1(x),
            self.branch2(x),
            self.branch3(x),
            self.branch4(x)
        ], dim=1)

In [16]:
class GoogLeNet_Modified(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        
        self.inception1 = InceptionBlock_Modified(64, 32, 32, 64, 32)
        self.inception2 = InceptionBlock_Modified(192, 64, 64, 128, 64)
        
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(384, 10)        
    def forward(self, x):
        x = self.conv1(x)
        x = self.inception1(x)
        x = self.inception2(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

In [17]:
def train_model(model, epochs=5, batch_size=64, lr=0.001):
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
    
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            preds = torch.argmax(model(xb), 1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    
    return correct / total

In [18]:
model = GoogLeNet_Modified()
accuracy = train_model(model, epochs=5)
print("Modified GoogLeNet Accuracy:", accuracy)

Modified GoogLeNet Accuracy: 0.9778571428571429
